# Loan Status Classifier: Predicting Loan Default Risk

**Author:** Peter Weber & Sean Nickerson    
**Course:** CPSC 322, Fall 2025  
**Date:** Fall 2025



This project implements machine learning to predict loan status (good vs. bad loans). We develop and compare multiple classification algorithms, including a custom Random Forest implementation, to identify loans at risk of default. Our models achieves strong predictive accuracy, helping financial institutions make better lending decisions.


# 1. Introduction

## 1.1 Dataset Description

This project uses the the [Lending Club Issued Loans dataset](https://www.kaggle.com/datasets/husainsb/lendingclub-issued-loans/data?select=lc_loan.csv) from Kaggle. The dataset contains information about loans issued by Lending Club, a lending platform, between 2007 and 2015. The original dataset contains 887,380 rows, but we use a truncated version with 180,000 rows and 74 attributes.

## 1.2 Classification Task

The primary goal of this project is to predict whether a loan will be **"Good"** (Fully Paid) or **"Bad"** (Charged Off, Late, or Default). This binary classification task is critical for financial institutions to assess loan risk before funding.

## 1.3 Key Findings

Our analysis reveals that:
- The dataset is highly imbalanced, with approximately 91% good loans and 9% bad loans
- Our custom Random Forest classifier performs competitively with sklearn implementations
- Feature selection and discretization improve model performance

## 1.4 Project Structure

This project implements:
1. **Custom Random Forest Classifier** (`MyRandomForestClassifier`) using custom requirements
3. **Feature selection and discretization** techniques
4. **Comprehensive evaluation** including parameter tuning for N, M, and F
5. **Comparison with single decision tree** baseline

We are using the truncated version of [kaggle dataset](https://www.kaggle.com/datasets/husainsb/lendingclub-issued-loans/data?select=lc_loan.csv), called "lc_loan.csv", which contains data from a company called Lending Club on all the loans they issued between 2007 and 2015, as well as the status of that loan. The original file is pretty large sitting at 887,380 rows, our truncated version has 180,000 rows and 74 attributes, we are looking to predict the status of each loan (whether it's deliquent or paid in full), and will do using the attribute loan_status.

Attribute Name | Description
------------ | -------------
loan_amnt|The Size Of The Loan
int_rate| Loan Interest Rate
dti| Debt To Income Ratio
annual_inc| Annual Income Of Borrower
term| How Long The Loan Lasts (12 months, 36 months, ect.)
grade| How the was ranked internally (A is better than B)
emp_length| How Long Borower Has Been Employed For
home_ownership| Does The Borrower Own A House
delinq_2yrs| Number Of Deliquencies In Past 2 Years
revol_util| Revolving Credit Utilization 

## Importing Relevant Code
Before we get started, we must import some code from the mysklearn package/folder

In [128]:
import numpy as np ##For random choices

import importlib
import os


import matplotlib.pyplot as plt

import mysklearn.myutils
importlib.reload(mysklearn.myutils)
import mysklearn.myutils as myutils

import mysklearn.mypytable
importlib.reload(mysklearn.mypytable)
from mysklearn.mypytable import MyPyTable 

import mysklearn.myclassifiers
importlib.reload(mysklearn.myclassifiers)
from mysklearn.myclassifiers import MyKNeighborsClassifier, MyDummyClassifier

import mysklearn.mytree
importlib.reload(mysklearn.mytree)
from mysklearn.mytree import MyDecisionTreeClassifier 

import mysklearn.myrandomforest
importlib.reload(mysklearn.myrandomforest)
from mysklearn.myrandomforest import MyRandomForestClassifier 

import mysklearn.myevaluation
importlib.reload(mysklearn.myevaluation)
import mysklearn.myevaluation as myevaluation

import mysklearn.myutils
importlib.reload(mysklearn.myutils)
import mysklearn.myutils as myutils

import mysklearn.plot_utils
importlib.reload(mysklearn.plot_utils)
import mysklearn.plot_utils as plot_utils

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')




## Loading And Examining Our Dataset

In [129]:
# Load the dataset
path = os.path.join("input_data", "lc_loan_truncated.csv")
dataset = MyPyTable().load_from_file(path)

In [130]:
dataset.print_shape()

printed_attributes = 0

print()
myutils.header_formatter('Dataset Columns',2)
for attribute in dataset.column_names:
    print(attribute, end=', ')
    printed_attributes += 1
    
    if printed_attributes == 8:
        print('\n')
        printed_attributes = 0


The Table Has 74 Attributes And 179999 instances

Dataset Columns
id, member_id, loan_amnt, funded_amnt, funded_amnt_inv, term, int_rate, installment, 

grade, sub_grade, emp_title, emp_length, home_ownership, annual_inc, verification_status, issue_d, 

loan_status, pymnt_plan, url, desc, purpose, title, zip_code, addr_state, 

dti, delinq_2yrs, earliest_cr_line, inq_last_6mths, mths_since_last_delinq, mths_since_last_record, open_acc, pub_rec, 

revol_bal, revol_util, total_acc, initial_list_status, out_prncp, out_prncp_inv, total_pymnt, total_pymnt_inv, 

total_rec_prncp, total_rec_int, total_rec_late_fee, recoveries, collection_recovery_fee, last_pymnt_d, last_pymnt_amnt, next_pymnt_d, 

last_credit_pull_d, collections_12_mths_ex_med, mths_since_last_major_derog, policy_code, application_type, annual_inc_joint, dti_joint, verification_status_joint, 

acc_now_delinq, tot_coll_amt, tot_cur_bal, open_acc_6m, open_il_6m, open_il_12m, open_il_24m, mths_since_rcnt_il, 

total_bal_il, il_u

74 attributes is far to many for our purposes of predicting if a loan is good or bad, we will have to drop many irrelevant columns.

Additionaly having roughly 180,000 instances is pretty problematic, our self-written code isn't as efficient as the standard sklearn approaches, so if we try to use them with this dataset as it currently is, our code will take forever to run, fortunately, their are a couple ways to cut down our dataset's size.

In [131]:
###Let's Check Out The Label Distribution For The Class We Want To Predict
#### We Want To See The Distribution Of class labels for our predicted class

loan_status = dataset.get_column('loan_status')
#print(loan_status)

label_distribution = myutils.count_label_distribution(loan_status)

# print(list(label_distribution.keys()))
# print(list(label_distribution.values()))
x = list(label_distribution.keys())
y = list(label_distribution.values())

for index in range(len(x)):
    if len(x[index]) > 22:
        x[index] = x[index][:22] + '...'

#plot_utils.bar_plot(x, y, 'Count Of Loan Status','Loan Label','Amount')


#### Analyzing Label Distribution

The end goal is to classify our loans as either 'good' or 'bad'. Looking at our label distribution,  
it seems like any loans that are 'Fully Paid', should be considered good, loans that are 'Charged Off', 'Late', 'Default' or 'In Grace Period' 
should be considered bad. With all other potential loan status like current being dropped from the dataset. We do this because with the other class labels, there is no easy way to tell if in the future they will end up being defaulted on or paid in full.

## With A Rough Sense Of Our Dataset It's Time For Cleaning!

### First Lets Remove Null Columns!

In [132]:
dataset.print_shape()
nulls_by_column = dataset.print_nan_by_column(null_print=False)


The Table Has 74 Attributes And 179999 instances


In [133]:
null_columns_to_drop = []
for column in nulls_by_column:
    if nulls_by_column[column] > (179999 // 1.5): ##Nulls are more than 66% of rows
        print(column, nulls_by_column[column])
        null_columns_to_drop.append(column)

mths_since_last_record 160214
mths_since_last_major_derog 152073
annual_inc_joint 179999
dti_joint 179999
verification_status_joint 179999
open_acc_6m 179999
open_il_6m 179999
open_il_12m 179999
open_il_24m 179999
mths_since_rcnt_il 179999
total_bal_il 179999
il_util 179999
open_rv_12m 179999
open_rv_24m 179999
max_bal_bc 179999
all_util 179999
inq_fi 179999
total_cu_tl 179999
inq_last_12m 179999


It seems like there are several columns that are completely, or nearly completely null values, we should drop them

In [134]:
### More Attribute Dropping:

# Drop irrelevant columns
columns_to_drop = [
    'id', 'member_id', 'url', 'desc', 'title', 'zip_code', 'addr_state',
    'issue_d', 'pymnt_plan', 'purpose', 'emp_title', 'verification_status',
    'initial_list_status', 'application_type', 'policy_code',
    'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d',
    'earliest_cr_line', 'sub_grade', 'funded_amnt', 'funded_amnt_inv'  ## sub_grade is redundant with grade, funded_amt + funded_amnt_inv are highly correlated
]                                                                      ## with loan amount

# Also drop columns that are post-loan
post_loan_columns = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt'
]

columns_to_drop = columns_to_drop + post_loan_columns + null_columns_to_drop

for attribute in columns_to_drop:
    dataset.drop_column(attribute)
dataset.print_shape()

The Table Has 23 Attributes And 179999 instances


In [135]:
##Next lets adjust our predicted class. Setting loan status to be good or bad,
##or alternatively dropping the row if it doesn't fit one of these two parameters

ls_i = dataset.column_names.index('loan_status')
rows_to_drop = []

bad_loans = ['Charged Off', 'Default', 'Late (31-120 days)', 'In Grace Period', 'Late (16-30 days)']

for r_i in range(len(dataset.data)):
    if dataset.data[r_i][ls_i] == 'Fully Paid' or dataset.data[r_i][ls_i] == 'Good Loan':
        dataset.data[r_i][ls_i] = 'Good Loan'
    elif dataset.data[r_i][ls_i] in bad_loans or dataset.data[r_i][ls_i] == 'Bad Loan':
        dataset.data[r_i][ls_i] = 'Bad Loan'
    else:
        rows_to_drop.append(r_i)
        
dataset.drop_rows(rows_to_drop)
dataset.print_shape()



The Table Has 23 Attributes And 114789 instances


In [136]:
loan_status = dataset.get_column('loan_status')
#print(loan_status)

label_distribution = myutils.count_label_distribution(loan_status)

# print(list(label_distribution.keys()))
# print(list(label_distribution.values()))
x = list(label_distribution.keys())
y = list(label_distribution.values())

for index in range(len(x)):
    if len(x[index]) > 22:
        x[index] = x[index][:22] + '...'

#plot_utils.bar_plot(x, y, 'Count Of Loan Status','Loan Label','Amount')

for value in label_distribution:
        print(value, label_distribution[value])

Good Loan 90635
Bad Loan 24154


In [137]:
#dataset.print_nan_by_column()

After our initial cleaning, it seems like 4 attributes still have many null values, I would like to keep these attributes because I think they are relevant to our classifiers. What I will end up doing is just dropping all rows where any attribute has a null value, before I do that, there is something I would like to check.

In [138]:
##For mths_since_last_delinq, I am curious what null means for a loan, if it just is a result of poor recording
##Or refers to situations where the user has never been deliquent.

column = dataset.get_column('mths_since_last_delinq')
label_distribution = myutils.count_label_distribution(column)
print('mths_since_last_delinq')
#print(list(label_distribution.items()))
print('Frequency of 0 within attribute:', label_distribution[0])


mths_since_last_delinq
Frequency of 0 within attribute: 501


Given the frequency of null values in the mths_since_last_delinq (first printed tuple), I am going to make an assumption . . . that the null values refer to situations where the person receiving the loan has never been deliquent, if this was not the case I would expect to see a specific value like 999999 for example appear quite frequently to represent all the people who have never been deliquent, but such a value didn't show up in the dataset from my quick scan.

With that being the case, I am going to substitute null values in mths_since_last_delinq with 1200 (100 years of non-deliquency) to represent never having been deliquent. 

In [139]:
deliq_index = dataset.column_names.index('mths_since_last_delinq')

for r_i in range(len(dataset.data)):
    if dataset.data[r_i][deliq_index] == '':
        dataset.data[r_i][deliq_index] = 1200

In [140]:
##I also want to do label encoding for two categorical variables so I don't have to modify my classifiers
##(Fortunately Sean's version of the DecisionTree + RandomForest can handle continuous data so this is totatly fine to do)

print(set(dataset.get_column('grade')))
print(set(dataset.get_column('home_ownership')))



{'E', 'C', 'B', 'D', 'G', 'F', 'A'}
{'NONE', 'RENT', 'MORTGAGE', 'OWN', 'OTHER'}


In [141]:
grade_labels = ['A','B','C','D','E','F','G']
grade_values = [100,90,80,70,60,50,40]

home_labels = ['OWN','MORTGAGE','RENT','NONE']
home_values = [40,30,20,10]

grade_index = dataset.column_names.index('grade')
home_index=  dataset.column_names.index('home_ownership')


for r_i in range(len(dataset.data)):
    dataset.data[r_i][grade_index] = myutils.label_encoder(dataset.data[r_i][grade_index], grade_labels, grade_values)
    dataset.data[r_i][home_index] = myutils.label_encoder(dataset.data[r_i][home_index], home_labels, home_values)


In [142]:
##Now to remove all remaining null rows:
myutils.header_formatter('Before Dropping Null Rows',2)
dataset.print_shape()
dataset.remove_rows_with_missing_values()

myutils.header_formatter('After Dropping Null Rows',2)
dataset.print_shape()


Before Dropping Null Rows
The Table Has 23 Attributes And 114789 instances
After Dropping Null Rows
The Table Has 23 Attributes And 73962 instances


In [143]:
##Some attributes record categorical data in the format of a string for example:
string_and_numeric_columns = ['term','emp_length']

for column in string_and_numeric_columns:
    column_data = dataset.get_column(column)
    for value in column_data:
        if not isinstance(value, (int, float)):
            print(f'The {column} attribute has the value \"{value}\", which is {type(value)}')
            break



The term attribute has the value " 36 months", which is <class 'str'>
The emp_length attribute has the value "10+ years", which is <class 'str'>


In [144]:
print(dataset.column_names)

['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'loan_status', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim']


In [145]:
###To Fix The Above Problem, We Will Just Strip Any Chars From The Value And Forcibly Convert It To Float

term_index = dataset.column_names.index('term')
emp_index = dataset.column_names.index('emp_length')


for r_i in range(len(dataset.data)):
    if not isinstance(dataset.data[r_i][term_index], (int, float)):
        value = dataset.data[r_i][term_index]        
        value = ''.join(char for char in value if char.isdigit() or char =='.' or char =='-')
        value = float(value)
        dataset.data[r_i][term_index] = value

    if not isinstance(dataset.data[r_i][emp_index], (int, float)):
        value = dataset.data[r_i][emp_index]
        value = ''.join(char for char in value if char.isdigit() or char =='.' or char =='-')
        value = float(value)
        dataset.data[r_i][emp_index] = value


In [146]:
for row in dataset.data[:200]:
    print(row[8])

Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Bad Loan
Good Loan
Good Loan
Good Loan
Good Loan
Bad Loan
Good Loan
Good Loan

## Now To Downsample:

We've got a dataset that is relatively clean, but it still has an incredible number of rows at ~70,000. This would make training and testing our models incredibly slow, so we are going to downsample and take 5,000 'Good Loans' and 5,000 'Bad Loans' to train and test our models.

In [147]:
##The first step in doing that is seperating our good and bad loans into two groups.

good_loans = []
bad_loans = []

loan_status_index = dataset.column_names.index('loan_status')
print(loan_status_index)

for row in dataset.data:
    if row[loan_status_index] == 'Good Loan':
        good_loans.append(row)
    elif row[loan_status_index] == 'Bad Loan':
        bad_loans.append(row)
    else:
        print('Problemo')

print('Good Loan Amount:', len(good_loans))
print('Bad Loan Amount:', len(bad_loans))

rng = np.random.default_rng(0)

good_loan_indices = range(len(good_loans))
bad_loan_indices = range(len(bad_loans))
                          
random_good_indices = rng.choice(good_loan_indices, size=5000, replace=False)
random_bad_indices = rng.choice(bad_loan_indices, size=5000, replace=False)

8
Good Loan Amount: 56466
Bad Loan Amount: 17496


In [148]:
downsampled_data = []

for row_index in range(5000):
    downsampled_data.append(dataset.data[random_good_indices[row_index]].copy())
    downsampled_data.append(dataset.data[random_bad_indices[row_index]].copy())


In [158]:
for row in downsampled_data:
    value = row[8]
    if value != 'Good Loan' and value != 'Bad Loan':
        print(value)
        print(len(row))
    elif len(row) != 23:
        print(row)

In [161]:
##I want to put loan status as the last attribute in our dataset for future ease of use:

loan_status = []

for r_i in range(len(downsampled_data)):
    value = downsampled_data[r_i][loan_status_index]
    loan_status.append(value)    
    downsampled_data[r_i].pop(loan_status_index)


In [163]:
for value in loan_status:
    if value != 'Good Loan' and value != 'Bad Loan':
        print(value)
        

In [164]:
for r_i in range(len(downsampled_data)):
    downsampled_data[r_i].append(loan_status[r_i])

In [165]:
for row in downsampled_data:
    value = row[-1]
    if value != 'Good Loan' and value != 'Bad Loan':
        print(value)
        print(len(row))
    elif len(row) != 23:
        print(row)

In [167]:

print(dataset.column_names)
print(downsampled_data[0])

print()

['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'loan_status', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim']
[10000, 36.0, 6.62, 307.04, 100, 1.0, 20, 148510, 7.62, 0, 0, 1200, 11, 0, 11253, 53.6, 26, 0, 0, 0, 55947, 21000, 'Good Loan']



## Now Its Finally Time To Run Our Classifiers

In [168]:
##First Seperate X From y

X = [row[:-1] for row in downsampled_data]
y = [row[-1] for row in downsampled_data]



In [169]:
for row in X:
    for value in row:
         if not isinstance(value, (int, float)):
             if value != 'Good Loan' and value != 'Bad Loan':
                 print(value)

In [ ]:
x_train, x_test, y_train, y_test = myevaluation.train_test_split(X, y, random_state = 0)

myDummy = MyDummyClassifier()
myKNN = MyKNeighborsClassifier()
myDTC = MyDecisionTreeClassifier()
myRF = MyRandomForestClassifier()

classifiers = [(myDummy, 'Dummy Classifier'), (myKNN, 'KNN Classifier'), (myDTC, 'Decision Tree Classifier'), (myRF, 'Random Forest Classifier')]

for classifier in classifiers:
    classifier[0].fit(x_train, y_train)
    y_pred = classifier[0].predict(x_test)
    myutils.header_formatter(f'{classifier[1]} Metrics', 2)
    
    myutils.accuracy_error_printer(y_test, y_pred)
    myutils.precision_recall_f1_printer(y_test, y_pred, ['Good Loan', 'Bad Loan'],'Good Loan')
    print('\n')
    myutils.header_formatter(f'{classifier[1]} Confusion Matrix', 2)
    
    cm = myevaluation.confusion_matrix(y_test, y_pred, ['Good Loan', 'Bad Loan'])
    myutils.formatted_confusion_matrix(cm, 'Loan Status', ['Good Loan', 'Bad Loan'])

Dummy Classifier Metrics
The classifier has an accuracy of: 0.76, and an error rate of: 0.24
The classifier had a precision of: 0.76, a recall of: 1.0, and f1_score of: 0.87


Dummy Classifier Confusion Matrix
Loan Status      Good Loan    Bad Loan    Total    Recognition %
-------------  -----------  ----------  -------  ---------------
Good Loan             2524           0     2524                1
Bad Loan               776           0      776                0
KNN Classifier Metrics
The classifier has an accuracy of: 0.71, and an error rate of: 0.29
The classifier had a precision of: 0.78, a recall of: 0.87, and f1_score of: 0.82


KNN Classifier Confusion Matrix
Loan Status      Good Loan    Bad Loan    Total    Recognition %
-------------  -----------  ----------  -------  ---------------
Good Loan             2192         332     2524             0.87
Bad Loan               624         152      776             0.2
